# 02 · Classifier: EfficientNet

**Goal:** train a 4-class brain tumor classifier (glioma, meningioma, notumor, pituitary) on the clean splits from `01_data_prep.ipynb`.

**Target:** accuracy and macro-F1 **≥ 95%** on the test split.

**How we stay honest:** we tune on **val only**. The test split stays closed (`RUN_TEST = False`) until the final model is picked, then it is run once.

**Presets** (all presets in `RUN_PRESETS` are trained one after another in a single run):

| Preset | Backbone | Image | Extras |
|---|---|---|---|
| `baseline` | EfficientNet-B0 | 224 px | light augmentation |
| `strong` | EfficientNet-B3 | 384 px | stronger augmentation, label smoothing, warmup |

**Brain crop (`CROP = True`):** Grad-CAM showed the first models also used the framing and look of each source dataset. Every scan is now cropped to the head and padded to a square before resizing, so all sources look the same to the model, and a random blur hides differences in sharpness.

**Run on Kaggle:** GPU on, Internet on, inputs `masoudnickparvar/brain-tumor-mri-dataset` + `splits.csv`. Best as *Save Version → Save & Run All*, so it runs in the background. Outputs land in `/kaggle/working`.

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset

SMOKE = os.environ.get("NEUROLENS_SMOKE") == "1"  # tiny local test run on fake data


def find_data_dir() -> Path:
    if "NEUROLENS_DATA" in os.environ:
        return Path(os.environ["NEUROLENS_DATA"])
    # Kaggle mounts each dataset under /kaggle/input; the folder holding Training/ is the root
    for p in Path("/kaggle/input").glob("**/Training"):
        return p.parent
    raise FileNotFoundError("Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset.")


def find_splits() -> Path:
    if "NEUROLENS_SPLITS" in os.environ:
        return Path(os.environ["NEUROLENS_SPLITS"])
    for p in [*Path("/kaggle/input").glob("**/splits.csv"), Path("../data/splits.csv")]:
        if p.exists():
            return p
    raise FileNotFoundError("splits.csv not found. Attach the output of 01_data_prep.ipynb.")


def crop_brain(img: np.ndarray, margin: float = 0.03) -> np.ndarray:
    """Crop an MRI slice to the head and pad it to a square, so every source has the same framing."""
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    # Low threshold: separates the head from the black background (Otsu would keep only a bright tumor)
    mask = (gray > max(10, 0.08 * np.percentile(gray, 99.5))).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))  # drop specks and thin text
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    if n < 2:
        return img
    x, y, w, h, area = stats[1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])]  # largest blob = the head
    if area < 0.05 * mask.size:
        return img  # nothing sensible found: keep the full image
    m = int(margin * max(w, h))
    x0, y0 = max(0, x - m), max(0, y - m)
    x1, y1 = min(img.shape[1], x + w + m), min(img.shape[0], y + h + m)
    crop = img[y0:y1, x0:x1]
    side = max(crop.shape[:2])  # pad to a square with black, keep the aspect ratio
    out = np.zeros((side, side, 3), dtype=img.dtype)
    oy, ox = (side - crop.shape[0]) // 2, (side - crop.shape[1]) // 2
    out[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop
    return out


DATA_DIR, SPLITS_CSV = find_data_dir(), find_splits()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| splits:", SPLITS_CSV)

## 1. Config

In [ ]:
RUN_PRESETS = ["baseline", "strong"]  # trained one after another
CROP = True       # crop every scan to the head (train and eval)
RUN_TEST = False  # open the test split only once, for the final model

PRESETS = {
    "baseline": dict(model="efficientnet_b0", img_size=224, batch_size=32, lr=3e-4,
                     label_smoothing=0.0, strong_aug=False, warmup_epochs=0, patience=6),
    "strong":   dict(model="efficientnet_b3", img_size=384, batch_size=24, lr=3e-4,
                     label_smoothing=0.1, strong_aug=True, warmup_epochs=1, patience=7),
}
COMMON = dict(crop=CROP, epochs=30, weight_decay=1e-4, num_workers=4, seed=42, pretrained=True)
SMOKE_OVERRIDES = dict(img_size=64, batch_size=8, epochs=2, num_workers=0, pretrained=False)
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]


def make_cfg(preset: str) -> dict:
    cfg = dict(preset=preset, **COMMON, **PRESETS[preset])
    if SMOKE:
        cfg.update(SMOKE_OVERRIDES)
    cfg["tag"] = preset + ("_crop" if cfg["crop"] else "")  # used in every output file name
    return cfg


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"  # mixed precision: faster and lighter on GPU
print("device:", device, "| amp:", use_amp, "| presets:", RUN_PRESETS, "| crop:", CROP)

## 2. Data

In [ ]:
df = pd.read_csv(SPLITS_CSV)
df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})
train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
df.groupby(["split", "label"]).size().unstack(fill_value=0)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet stats, to match the pretrained backbone


def make_transforms(cfg: dict):
    S = cfg["img_size"]
    if cfg["strong_aug"]:
        aug = [
            A.HorizontalFlip(p=0.5),  # brains are roughly left-right symmetric
            A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-20, 20), p=0.8),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
            A.RandomGamma(p=0.3),        # the dataset has the same scans with other contrast curves
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),  # sources differ in sharpness: don't let that be a clue
            A.GaussNoise(p=0.2),
            A.CoarseDropout(p=0.3),      # small cut-out holes: don't rely on one spot
        ]
    else:
        aug = [
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        ]
    train_tf = A.Compose([A.Resize(S, S), *aug, A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    eval_tf = A.Compose([A.Resize(S, S), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    return train_tf, eval_tf


class MRIDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tf: A.Compose, crop: bool):
        self.paths = [DATA_DIR / p for p in frame.path]
        self.ys = frame.y.tolist()
        self.tf, self.crop = tf, crop

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert("RGB"))  # grayscale MRI -> 3 channels for the pretrained net
        if self.crop:
            img = crop_brain(img)
        return self.tf(image=img)["image"], self.ys[i]


def make_loader(frame, tf, cfg, shuffle):
    return DataLoader(MRIDataset(frame, tf, cfg["crop"]), batch_size=cfg["batch_size"], shuffle=shuffle,
                      num_workers=cfg["num_workers"], pin_memory=use_amp, drop_last=shuffle)

### Check a few augmented, cropped training images

In [ ]:
cfg = make_cfg(RUN_PRESETS[-1])
x, y = next(iter(make_loader(train_df, make_transforms(cfg)[0], cfg, shuffle=True)))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img, label in zip(axes, x[:6], y[:6]):
    img = img.permute(1, 2, 0).numpy() * STD + MEAN  # undo normalize for display
    ax.imshow(img.clip(0, 1))
    ax.set_title(CLASSES[label])
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "augmented_examples.png", dpi=80)
plt.show()

## 3. Training and evaluation functions

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    """One pass over the loader (trains if an optimizer is given). Returns loss, accuracy, macro-F1, labels, preds, probs."""
    train = optimizer is not None
    model.train(train)
    total_loss, ys, probs = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)  # loss in float32: fp16 logits can overflow to inf/NaN
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * len(y)
            ys += y.tolist()
            probs.append(logits.float().softmax(1).detach().cpu())
    probs = torch.cat(probs).numpy()
    preds = probs.argmax(1)
    return total_loss / len(ys), accuracy_score(ys, preds), f1_score(ys, preds, average="macro"), np.array(ys), preds, probs


def train(cfg: dict):
    """Train one preset with early stopping on val macro-F1. Returns the best model and the history."""
    seed_everything(cfg["seed"])
    train_tf, eval_tf = make_transforms(cfg)
    train_loader = make_loader(train_df, train_tf, cfg, shuffle=True)
    val_loader = make_loader(val_df, eval_tf, cfg, shuffle=False)

    model = timm.create_model(cfg["model"], pretrained=cfg["pretrained"], num_classes=len(CLASSES)).to(device)
    # Class weights so a bigger class doesn't dominate; label smoothing keeps the model from being over-confident
    counts = train_df.y.value_counts().sort_index().to_numpy()
    weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=cfg["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    def lr_factor(epoch: int) -> float:  # linear warmup, then cosine decay to zero
        w, total = cfg["warmup_epochs"], cfg["epochs"]
        if epoch < w:
            return (epoch + 1) / (w + 1)
        return 0.5 * (1 + np.cos(np.pi * (epoch - w) / max(1, total - w)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    ckpt = OUT_DIR / f"classifier_{cfg['tag']}.pt"
    print(f"--- {cfg['tag']}: {cfg['model']}, {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

    best_f1, bad_epochs, history = -1.0, 0, []
    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()
        tr_loss, tr_acc, *_ = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_acc, va_f1, *_ = run_epoch(model, val_loader, criterion)
        scheduler.step()
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss, val_acc=va_acc, val_f1=va_f1))
        print(f"epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f} f1 {va_f1:.3f} | {time.time() - t0:.0f}s")
        if va_f1 > best_f1:
            best_f1, bad_epochs, cfg["best_epoch"] = va_f1, 0, epoch
            torch.save(model.state_dict(), ckpt)
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                print(f"early stop: no val F1 gain for {cfg['patience']} epochs")
                break

    model.load_state_dict(torch.load(ckpt, map_location=device))
    hist = pd.DataFrame(history)
    hist.to_csv(OUT_DIR / f"history_{cfg['tag']}.csv", index=False)
    return model, criterion, eval_tf, hist

In [ ]:
def plot_curves(hist: pd.DataFrame, tag: str) -> None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist.epoch, hist.train_loss, label="train")
    ax1.plot(hist.epoch, hist.val_loss, label="val")
    ax1.set_title(f"loss · {tag}")
    ax1.legend()
    ax2.plot(hist.epoch, hist.train_acc, label="train acc")
    ax2.plot(hist.epoch, hist.val_acc, label="val acc")
    ax2.plot(hist.epoch, hist.val_f1, label="val macro-F1")
    ax2.set_title(f"metrics · {tag}")
    ax2.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"curves_{tag}.png", dpi=80)
    plt.show()


def report(model, criterion, eval_tf, cfg, name: str, frame: pd.DataFrame) -> dict:
    loss, acc, f1, ys, preds, probs = run_epoch(model, make_loader(frame, eval_tf, cfg, shuffle=False), criterion)
    tag = cfg["tag"]
    print(f"{tag} {name}: acc {acc:.4f} | macro-F1 {f1:.4f}")
    print(classification_report(ys, preds, labels=range(len(CLASSES)), target_names=CLASSES, digits=4, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(ys, preds, labels=range(len(CLASSES)), display_labels=CLASSES, cmap="Blues", colorbar=False)
    plt.title(f"{name} confusion matrix · {tag}")
    plt.savefig(OUT_DIR / f"confusion_{name}_{tag}.png", dpi=80, bbox_inches="tight")
    plt.show()

    # Wrong predictions, most confident first: these show what the model still gets wrong
    wrong = np.nonzero(preds != ys)[0]
    wrong = wrong[np.argsort(-probs[wrong].max(1))][:12]
    if len(wrong):
        fig, axes = plt.subplots(2, 6, figsize=(18, 6), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, i in zip(axes.flat, wrong):
            img = np.array(Image.open(DATA_DIR / frame.path[i]).convert("RGB"))
            ax.imshow(crop_brain(img) if cfg["crop"] else img)
            ax.set_title(f"true {CLASSES[ys[i]]}\npred {CLASSES[preds[i]]} ({probs[i].max():.0%})", fontsize=9)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"errors_{name}_{tag}.png", dpi=70)
        plt.show()
    return dict(acc=round(acc, 4), macro_f1=round(f1, 4), errors=int((preds != ys).sum()))

## 4. Train every preset, evaluate on val, save

In [ ]:
summary = {}
for preset in RUN_PRESETS:
    cfg = make_cfg(preset)
    model, criterion, eval_tf, hist = train(cfg)
    plot_curves(hist, cfg["tag"])
    val_metrics = report(model, criterion, eval_tf, cfg, "val", val_df)
    test_metrics = report(model, criterion, eval_tf, cfg, "test", test_df) if RUN_TEST else None
    metrics = dict(config=cfg, best_epoch=cfg["best_epoch"], val=val_metrics, test=test_metrics,
                   n_train=len(train_df), n_val=len(val_df), n_test=len(test_df))
    (OUT_DIR / f"metrics_{cfg['tag']}.json").write_text(json.dumps(metrics, indent=2))
    summary[cfg["tag"]] = val_metrics
    del model
    torch.cuda.empty_cache()

pd.DataFrame(summary).T

**Next:** *Output → Download all*. Weights are `classifier_<tag>.pt` (too big for git, kept locally in `ml/models/`); metrics, history, curves, confusion matrices and error grids go to `ml/results/`.